In [ ]:
import numpy as np
import time
from pynq import allocate, MMIO, Overlay

In [ ]:
ol = Overlay("cracked_design.bit")
print(ol.ip_dict.keys())

In [ ]:
# --- Parameters & Addresses ---
D = 17
NUM_SAMPLES = 7375
IP_BASE   = 0x43C00000
DMA_BASE  = 0x40400000
BRAM_BASE = 0x40000000

# --- Setup MMIO ---
ip_ctrl = MMIO(IP_BASE, 64)
dma     = MMIO(DMA_BASE, 64)
bram    = MMIO(BRAM_BASE, 8192) # 8192 handles the stretched AXI memory

In [ ]:
# ---  Data Generation --- #

test_x = np.random.uniform(0.5, 1.5, (NUM_SAMPLES, D))
test_y = np.random.uniform(0.5, 1.5, NUM_SAMPLES)

test_x_quantized = np.round(test_x * 16).astype(np.int16)
test_y_quantized = np.round(test_y * 16).astype(np.int16)

mem_in = allocate(shape=(NUM_SAMPLES, 32), dtype=np.int16)

# ---- Vectorized Data Packing ---- #

try:

    pack_start = time.time()

    mem_in[:, 0] = test_y_quantized
    mem_in[:, 1:18] = test_x_quantized
    mem_in[:, 18:32] = 0
    mem_in.flush()
    pack_time = time.time() - pack_start

# ---- Hardware Execution ---- #

    ip_ctrl.write(0x00, 0x02) # Clear
    ip_ctrl.write(0x00, 0x00)
    ip_ctrl.write(0x08, NUM_SAMPLES) # Set samples
    ip_ctrl.write(0x00, 0x01) # Start
    ip_ctrl.write(0x00, 0x00)

    dma_start = time.time()
    dma.write(0x00, 0x01) # DMA MM2S Control Run
    dma.write(0x18, mem_in.device_address)
    dma.write(0x28, NUM_SAMPLES * 64) # Length in bytes
    dma_setup_time = time.time() - dma_start

    # Bypass STAT check and allow silicon to compute
    time.sleep(0.1) 

    # --- Result Retrieval (Clean 1:1 Mapping) --- #

    bram_start = time.time()
    bram_array = bram.array.view(np.int64)

    # 1. Extract Atb 
    hw_atb_raw = np.copy(bram_array[0 : D])

    # 2. Extract AtA 
    ata_start_idx = D
    ata_end_idx = ata_start_idx + int(D*(D+1)/2)
    ata_lower_raw = np.copy(bram_array[ata_start_idx : ata_end_idx])

    # 3. Vectorized Matrix Reconstruction
    hw_ata_raw = np.zeros((D, D), dtype=np.int64)
    row_idx, col_idx = np.tril_indices(D)
    hw_ata_raw[row_idx, col_idx] = ata_lower_raw
    hw_ata_raw[col_idx, row_idx] = ata_lower_raw

    bram_time = time.time() - bram_start

    # ---- Software Simulation ---- #

    sw_start_time = time.time()
    sw_x = test_x_quantized.astype(np.int64)
    sw_y = test_y_quantized.astype(np.int64)

    sw_ata_final = sw_x.T @ sw_x
    sw_atb_final = sw_x.T @ sw_y
    sw_time = time.time() - sw_start_time

    # ---- Match checking ---- #

    ata_match = np.allclose(hw_ata_raw, sw_ata_final, atol=0)
    atb_match = np.allclose(hw_atb_raw, sw_atb_final, atol=0)

    hw_cycles = ip_ctrl.read(0x0C) 
    hw_silicon_time = (hw_cycles / 100_000_000.0)

    print(f"\n--- Final Validation ---")
    print(f"AtA Match: {ata_match}")
    print(f"Atb Match: {atb_match}")

    if not ata_match:
        mismatches = np.argwhere(hw_ata_raw != sw_ata_final)
        print(f"-> Mismatches found: {len(mismatches)}")

    hw_cycles = ip_ctrl.read(0x0C) 
    print(f"AtA Sample (0,0) HW [Raw]: {hw_ata_raw[0,0]} | SW [Raw]: {sw_ata_final[0,0]}") 
    print(f"Atb Sample (0)   HW [Raw]: {hw_atb_raw[0]} | SW [Raw]: {sw_atb_final[0]}")
    
    print(f"\n--- Detailed Timing Breakdown ---")
    print(f"1. Array Packing & Flush  : {pack_time:.6f} seconds")
    print(f"2. DMA Register Setup     : {dma_setup_time:.6f} seconds")
    print(f"3. Pure Silicon Hardware  : {hw_silicon_time:.6f} seconds ({hw_cycles} cycles)")
    print(f"4. BRAM Read & Rebuild    : {bram_time:.6f} seconds")
    print(f"5. Total Software Time    : {sw_time:.6f} seconds")


finally:
        mem_in.freebuffer()

In [ ]:
# ---- End-to-End Comparison ---- 

total_hw_time = pack_time + dma_setup_time + hw_silicon_time + bram_time

print(f"\n--- Total End-to-End System Comparison ---")
print(f"Total Software Time (NumPy) : {sw_time:.6f} seconds")
print(f"Total Hardware Time (System): {total_hw_time:.6f} seconds")

if total_hw_time < sw_time:
    speedup = sw_time / total_hw_time
    print(f"-> OVERALL VERDICT: The Full Hardware System is {speedup:.2f}x FASTER than Software.")
else:
    slowdown = total_hw_time / sw_time
    print(f"-> OVERALL VERDICT: The Full Hardware System is {slowdown:.2f}x SLOWER than Software.")